# FloodEye: huấn luyện trên Kaggle

Notebook chỉ làm ba việc: lấy mã ở một commit cố định, chuẩn bị dữ liệu bằng script của kho, gọi `training/train.py`. Mọi logic nằm trong kho mã.

Trước khi chạy: bật **GPU T4** và **Internet** ở khung Settings (Internet cần tài khoản đã xác minh số điện thoại).

Cách dùng:
1. Để `MODE = "smoke"`, bấm **Run All** trong phiên tương tác (khoảng 15 phút). Lần chạy thử bật mosaic như huấn luyện thật, trên 1/4 dữ liệu, để kiểm tra bộ nhớ GPU. Xong thì **Stop session** để không tốn hạn mức.
2. Đổi `MODE = "train"` rồi **Save Version → Save & Run All (Commit)**. Lần chạy này tiếp tục khi đóng trình duyệt; một lần chạy nền bị cắt ở 12 giờ, nên `EPOCHS` phải giữ cả lần chạy dưới khoảng 10 giờ.

In [ ]:
COMMIT = "e26a81bfb6f169d7c46910086aa2e495bf8c3d84"  # mã commit đầy đủ; đổi khi mã huấn luyện thay đổi
CONFIG = "configs/experiments/E2_s0.yaml"
MODE = "smoke"   # "smoke": 1 epoch trên 1/4 dữ liệu, bật mosaic; "train": huấn luyện thật
EPOCHS = 60      # None: theo file cấu hình (100); 60 epoch ≈ 8–9 giờ trên một T4 (đo 21/10)

In [ ]:
import os

if not os.path.isdir("/kaggle/working/floodeye"):
    !git clone -q https://github.com/huzy179/uav-rescue-qa.git /kaggle/working/floodeye
%cd /kaggle/working/floodeye
!git fetch -q && git checkout -q {COMMIT} && git log -1 --oneline
!pip install -q -e ".[dev]"

Các dòng `ERROR: pip's dependency resolver ...` ở ô trên bỏ qua được: chúng nói về gói Kaggle cài sẵn mà dự án không dùng (`bigframes`, `google-colab`, `cudf`, ...). Ô dưới kiểm tra những gói dự án thật sự cần.

In [ ]:
import torch, cv2, ultralytics

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| số GPU", torch.cuda.device_count())
print("opencv", cv2.__version__, "| ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Chưa có GPU: bật Accelerator GPU T4 ở khung Settings"

## Dữ liệu

VisDrone train và val, chuyển về một lớp `person`. Đã tải rồi thì bỏ qua bước tải.

In [ ]:
%%bash
set -e
mkdir -p data/raw/visdrone
for split in train val; do
  if [ ! -d data/raw/visdrone/VisDrone2019-DET-$split ]; then
    (cd data/raw/visdrone && curl -sLO https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-$split.zip \
      && unzip -q VisDrone2019-DET-$split.zip && rm VisDrone2019-DET-$split.zip)
  fi
  python tools/convert_visdrone.py --src data/raw/visdrone/VisDrone2019-DET-$split --split $split
done

C2A (cho `E2b_*`): thêm dataset Kaggle **c2a-dataset** (của rgbnihal) bằng *Add Input*. Ô dưới tìm thư mục có `train/images` trong `/kaggle/input`, rồi chuyển về một lớp person bằng `tools/convert_yolo_subset.py`. Không thêm input thì ô này bỏ qua, `E2_s0` vẫn chạy được.

In [ ]:
from pathlib import Path

roots = sorted({p.parent.parent for p in Path('/kaggle/input').glob('**/train/images') if 'c2a' in str(p).lower()})
if not roots:
    print('Không có C2A trong /kaggle/input: bỏ qua (chỉ cần cho cấu hình E2b_*)')
elif len(roots) > 1:
    raise RuntimeError(f'Nhiều thư mục C2A, chọn một: {roots}')
else:
    print('C2A:', roots[0])
    !python tools/convert_yolo_subset.py --src {roots[0]} --dst data/processed/c2a --class-names human --keep human --splits train val test


## Huấn luyện

Chỉ dùng một GPU (`CUDA_VISIBLE_DEVICES=0`). Với hai GPU hiện diện, ultralytics chia 4 CPU của Kaggle cho số GPU, nên chỉ còn 2 tiến trình đọc dữ liệu dù chỉ huấn luyện trên một GPU; ẩn GPU thứ hai thì được 4.

`smoke` ghi vào `runs/train/smoke` để không lẫn với kết quả thật. Nếu log có dòng `out of memory ... Reducing to batch=4` thì dừng lại: batch bị tự giảm làm sai cấu hình thí nghiệm.

Phiên bị ngắt thì chạy lại ô dưới với `--resume` thêm vào cuối lệnh, sau khi đã chép `runs/train/` của phiên trước về đúng chỗ.

In [ ]:
import time

if MODE == "smoke":
    args = "--epochs 1 --close-mosaic 0 --fraction 0.25 --output runs/train/smoke"
elif MODE == "train":
    args = f"--epochs {EPOCHS}" if EPOCHS else ""
else:
    raise ValueError(f"MODE phải là 'smoke' hoặc 'train', nhận được {MODE!r}")

start = time.perf_counter()
!CUDA_VISIBLE_DEVICES=0 python training/train.py --config {CONFIG} --device 0 {args}
print(f"Thời gian chạy ({MODE}): {(time.perf_counter() - start) / 60:.1f} phút")

## Dọn dẹp

Xoá dữ liệu để phần Output của notebook chỉ còn mã và `runs/train/` (trọng số, `results.csv`, `run_config.json`). Lưu Output thành Kaggle Dataset có phiên bản rồi tải về `runs/train/` trên máy.

In [ ]:
!rm -rf data